# DAY 3 - Dataset Profiling

In [1]:
pip install pandas openpyxl matplotlib

Note: you may need to restart the kernel to use updated packages.


### Load Datasets

In [2]:
import pandas as pd

# Load datasets
interactions = pd.read_csv("../data/raw/llm_system_interactions.csv")
prompts = pd.read_csv("../data/raw/llm_system_prompts_lookup.csv")
sessions = pd.read_csv("../data/raw/llm_system_sessions_summary.csv")
users = pd.read_csv("../data/raw/llm_system_users_summary.csv")

print("Datasets Loaded Successfully!")

Datasets Loaded Successfully!


### Profile Row and Column Counts per File

In [3]:
datasets = {
    "Interactions": interactions,
    "Prompts": prompts,
    "Sessions": sessions,
    "Users": users
}

summary = []

for name, df in datasets.items():
    summary.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1]
    })

summary_df = pd.DataFrame(summary)
summary_df

,Dataset,Rows,Columns
0,Interactions,9000,55
1,Prompts,36,14
2,Sessions,1595,37
3,Users,438,21


### Inspect Data Types for All Columns

In [4]:
for name, df in datasets.items():
    print("=" * 70)
    print(name)
    print("=" * 70)
    print(df.dtypes)
    print()

Interactions
interaction_id                         object
session_id                             object
user_id                                object
timestamp_utc                          object
split                                  object
channel                                object
use_case                               object
country_code                           object
region                                 object
account_tier                           object
segment                                object
instruction_template                   object
model_provider                         object
model_name                             object
temperature                           float64
top_p                                 float64
max_tokens                              int64
prompt_tokens                           int64
completion_tokens                       int64
total_tokens                            int64
latency_ms                              int64
cost_usd             

### Log Missing Value Counts per Column

In [5]:
for name, df in datasets.items():
    print("=" * 70)
    print(name)
    print("=" * 70)

    missing = df.isnull().sum()
    missing = missing[missing > 0]

    if len(missing) == 0:
        print("No missing values found.\n")
    else:
        print(missing.sort_values(ascending=False))
        print()

Interactions
user_feedback_score    4120
dtype: int64

Prompts
No missing values found.

Sessions
csat_mean              63
requests_per_minute    35
dtype: int64

Users
avg_csat    3
dtype: int64



### Identify Duplicate Records using interaction_id, session_id, prompt_id, and user_id

In [6]:
print("Duplicate Interaction IDs:",
      interactions["interaction_id"].duplicated().sum())

print("Duplicate Session IDs in Sessions:",
      sessions["session_id"].duplicated().sum())

print("Duplicate Prompt IDs:",
      prompts["prompt_id"].duplicated().sum())

print("Duplicate User IDs:",
      users["user_id"].duplicated().sum())

Duplicate Interaction IDs: 0
Duplicate Session IDs in Sessions: 0
Duplicate Prompt IDs: 0
Duplicate User IDs: 0


# DAY 4 - Data Quality Validation

### Detect Data Type Issues
* Validate numeric-as-text columns
* Validate malformed dates in `timestamp_utc`
* Validate `latency_timeout_flag`

In [8]:
import pandas as pd

datasets = {
    "Interactions": interactions,
    "Prompts": prompts,
    "Sessions": sessions,
    "Users": users
}

print("="*80)
print("DATA TYPE VALIDATION")
print("="*80)

for name, df in datasets.items():
    print(f"\n{name}")
    print("-"*80)

    # Timestamp validation
    if "timestamp_utc" in df.columns:
        invalid_dates = pd.to_datetime(
            df["timestamp_utc"],
            errors="coerce"
        ).isna().sum()

        print(f"Invalid timestamp_utc values : {invalid_dates}")

    # Numeric columns stored as text
    for col in df.columns:
        if df[col].dtype == "object":

            try:
                converted = pd.to_numeric(df[col], errors="coerce")

                if converted.notna().sum() > 0:
                    print(f"Possible numeric stored as text -> {col}")

            except:
                pass

    # latency_timeout_flag validation
    if "latency_timeout_flag" in df.columns:

        print(
            "latency_timeout_flag values :",
            df["latency_timeout_flag"].unique()
        )

DATA TYPE VALIDATION

Interactions
--------------------------------------------------------------------------------
Invalid timestamp_utc values : 0
latency_timeout_flag values : [False  True]

Prompts
--------------------------------------------------------------------------------

Sessions
--------------------------------------------------------------------------------

Users
--------------------------------------------------------------------------------


### Detect Invalid or Out-of-Range Values
* Validate `country_code`
* Validate `account_tier`
* Validate `total_tokens`

In [11]:
print("=" * 80)
print("INVALID / OUT-OF-RANGE VALUE CHECK")
print("=" * 80)

# -------------------------
# Country Codes
# -------------------------
if "country_code" in interactions.columns:
    invalid_country = interactions[
        interactions["country_code"].str.len() != 2
    ]
    print(f"\nInvalid country_code values : {len(invalid_country)}")

# -------------------------
# Account Tier
# -------------------------
valid_tiers = ["free", "pro", "enterprise"]

if "account_tier" in interactions.columns:
    invalid_tier = interactions[
        ~interactions["account_tier"].isin(valid_tiers)
    ]

    print(f"Invalid account_tier values : {len(invalid_tier)}")

# -------------------------
# Total Tokens
# -------------------------
if "total_tokens" in interactions.columns:
    invalid_tokens = interactions[
        interactions["total_tokens"] < 0
    ]

    print(f"Negative total_tokens : {len(invalid_tokens)}")

INVALID / OUT-OF-RANGE VALUE CHECK

Invalid country_code values : 0
Invalid account_tier values : 0
Negative total_tokens : 0


### Detect Inconsistent Categories
* Check `channel`
* Check `region`
* Check `segment`

In [10]:
categorical_columns = [
    "account_tier",
    "channel",
    "region",
    "segment"
]

for col in categorical_columns:
    if col in interactions.columns:
        print("=" * 80)
        print(col.upper())
        print("=" * 80)
        print(interactions[col].value_counts(dropna=False))
        print()

ACCOUNT_TIER
account_tier
free          5007
pro           2532
enterprise    1461
Name: count, dtype: int64

CHANNEL
channel
web_app          3644
api              2260
mobile_app       1215
internal_tool     943
slack             938
Name: count, dtype: int64

REGION
region
EMEA    4234
AMER    2781
APAC    1985
Name: count, dtype: int64

SEGMENT
segment
individual         4221
team               3087
enterprise_team    1692
Name: count, dtype: int64



### Day 4 Data Quality Summary

In [12]:
print("=" * 80)
print("DAY 4 DATA QUALITY SUMMARY")
print("=" * 80)

print("""
✔ No malformed timestamp values detected.

✔ latency_timeout_flag contains only valid boolean values.

✔ All country_code values follow the expected two-character format.

✔ account_tier contains valid categories:
   - free
   - pro
   - enterprise

✔ No negative values found in total_tokens.

✔ channel categories are consistent:
   - web_app
   - api
   - mobile_app
   - internal_tool
   - slack

✔ region categories are consistent:
   - AMER
   - EMEA
   - APAC

✔ segment categories are consistent:
   - individual
   - team
   - enterprise_team

Overall Result:
No significant data quality issues were detected. The datasets are suitable for the data cleaning and exploratory analysis phase.
""")

DAY 4 DATA QUALITY SUMMARY

✔ No malformed timestamp values detected.

✔ latency_timeout_flag contains only valid boolean values.

✔ All country_code values follow the expected two-character format.

✔ account_tier contains valid categories:
   - free
   - pro
   - enterprise

✔ No negative values found in total_tokens.

✔ channel categories are consistent:
   - web_app
   - api
   - mobile_app
   - internal_tool
   - slack

✔ region categories are consistent:
   - AMER
   - EMEA
   - APAC

✔ segment categories are consistent:
   - individual
   - team
   - enterprise_team

Overall Result:
No significant data quality issues were detected. The datasets are suitable for the data cleaning and exploratory analysis phase.

